# Use Case 7: Fine-Tune MobileNetV2

This notebook unfreezes the last part of a pretrained network and trains it with a very small learning rate.

Each code cell is preceded by Markdown that explains what the step does, why it is needed, and which deep-learning concept is being demonstrated.


## Step 1: Import libraries

MobileNetV2 utilities are imported.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Rescaling,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D,
)

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


## Step 2: Load the scene dataset

Moon, space and landscape images are included.

In [ ]:
DATASET_PATH = "../datasets/03_scene_classification"
IMAGE_SIZE = (160, 160)
BATCH_SIZE = 16

DATASET_DIR = Path(DATASET_PATH)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names

print("Classes:", class_names)
print("Training batches:", len(train_ds))
print("Validation batches:", len(val_ds))


## Step 3: Train the frozen model first

Initial training adjusts only the new output layers.

In [ ]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(160, 160, 3),
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(160, 160, 3))
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(len(class_names), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=4,
)


## Step 4: Unfreeze only the final layers

Keeping most layers frozen preserves general visual knowledge.

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

print(
    "Trainable base layers:",
    sum(int(layer.trainable) for layer in base_model.layers)
)


## Step 5: Fine-tune with a small learning rate

A small learning rate prevents large destructive weight updates.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.00001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
)


## Step 6: Evaluate

The final model is evaluated after fine-tuning.

In [ ]:
loss, accuracy = model.evaluate(val_ds)

print("Validation loss:", round(loss, 4))
print("Validation accuracy:", round(accuracy, 4))
